# ⌨️ BioEcho v2 — Notebook 4: Keystroke Dynamics (Typing) Model
**Detects:** Cognitive decline · Motor neuron degradation · Mental health trajectory

| Feature | v2 Upgrade |
|---|---|
| Keystroke encoder | **RoPE positional encoding + SwiGLU (LLaMA-style)** |
| Optimizer | AdamW-8bit |
| Precision | BF16 |
| Compile | torch.compile reduce-overhead |
| Augmentation | Mixup + temporal jitter + CutMix |
| Loss | Gaussian NLL + Kendall-Gal task weighting |
| Calibration | ECE + reliability diagrams |
| Targets | 8 (added bp_systolic, hrv_sdnn) |
| Export | FP32 → INT8 ONNX (RTX 3050 ready) |
| Autosave | Full training state every epoch |

**RoPE:** Rotary Position Embedding — encodes temporal position of each keystroke.
**SwiGLU:** Swish-gated linear unit — feedforward activation (same as LLaMA).

**Kaggle:** saroopmandal / KGAT_2a969618d36d94f56d0989908ec94774

In [ ]:
import json; from pathlib import Path
kd=Path.home()/'.kaggle'; kd.mkdir(exist_ok=True)
cp=kd/'kaggle.json'
# On Kaggle, credentials are auto-injected. No hardcoded key needed.
# If running locally, set KAGGLE_USERNAME and KAGGLE_KEY env vars.
import os
json.dump({'username': os.environ.get('KAGGLE_USERNAME', 'saroopmandal'),
           'key': os.environ.get('KAGGLE_KEY', '')}, open(cp, 'w'))
cp.chmod(0o600); print('✅ Kaggle creds')

In [ ]:
import subprocess
def run(cmd,timeout=900):
    r=subprocess.run(cmd,shell=True,capture_output=True,text=True,timeout=timeout)
    if r.returncode!=0 and r.stderr: print(f'[{cmd[:40]}]:{r.stderr[:200]}')
    return r
run('pip install -q --upgrade pip')
run('pip install -q torch --index-url https://download.pytorch.org/whl/cu121',timeout=600)
run('pip install -q bitsandbytes>=0.43.0')
run('pip install -q scipy scikit-learn einops rich matplotlib seaborn pandas torchmetrics')
run('pip install -q onnx onnxruntime-gpu')
from onnxruntime.quantization import quantize_dynamic, QuantType
print('✅ Done')

In [ ]:
import os,gc,time,warnings,random,math,urllib.request,zipfile
from copy import deepcopy
from dataclasses import dataclass,field
from typing import List
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score,average_precision_score,f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from rich.console import Console; from rich.table import Table
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
# GradScaler: use torch.amp.GradScaler (torch.cuda.amp is deprecated)
import bitsandbytes as bnb
import onnx, onnxruntime as ort

warnings.filterwarnings('ignore')
console=Console()
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
NUM_GPUS=torch.cuda.device_count()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE=torch.float16
console.print(f'[bold green]GPUs:{NUM_GPUS}|BF16|{DEVICE}[/]')

In [ ]:
@dataclass
class KeyConfig:
    data_dir: str='/kaggle/working/bioecho/keystroke/data'
    ckpt_dir: str='/kaggle/working/bioecho/keystroke/checkpoints'
    out_dir:  str='/kaggle/working/bioecho/keystroke/outputs'
    seq_len:  int=128
    n_feat:   int=7
    # LLaMA-style encoder
    d_model:  int=256    # transformer model dim
    n_heads:  int=8      # attention heads
    n_layers: int=4      # transformer layers
    ffn_mult: float=2.67 # SwiGLU ffn multiplier (2/3 * 4 = 2.67)
    rope_base: int=10000 # RoPE base frequency
    embed_dim: int=256   # output embedding
    dropout:  float=0.2
    batch_size: int=256
    grad_accum: int=1
    epochs:   int=15
    lr:       float=1e-3
    weight_decay: float=1e-4
    grad_clip: float=1.0
    patience: int=5
    ema_decay: float=0.9995
    n_folds:  int=3

C=KeyConfig()
for d in [C.data_dir,C.ckpt_dir,C.out_dir]: Path(d).mkdir(parents=True,exist_ok=True)

BINARY_TASKS=['cognitive_decline','motor_degradation']
REGRESS_TASKS=['severity','stress_level','bp_systolic','hrv_sdnn','cognitive_load']
ALL_TASKS=BINARY_TASKS+REGRESS_TASKS
console.print(f'[cyan]LLaMA-style: RoPE+SwiGLU | d_model={C.d_model} | layers={C.n_layers}[/]')

In [ ]:
CKPT_DIR=Path(C.ckpt_dir)
def save_ckpt(epoch,ms,ema_s,opt,sch,scl,hist,best):
    p=CKPT_DIR/f'key_ep{epoch:03d}.pt'
    torch.save({'epoch':epoch,'model_state':ms,'ema_shadow':ema_s,'opt':opt,'sch':sch,'scl':scl,'hist':hist,'best':best},p)
    old=sorted(CKPT_DIR.glob('key_ep*.pt'),key=lambda x:int(x.stem[6:]))[:-2]
    for o in old: o.unlink(missing_ok=True)
def find_latest():
    ckpts=sorted(CKPT_DIR.glob('key_ep*.pt'),key=lambda x:int(x.stem[6:]))
    return ckpts[-1] if ckpts else None
latest=find_latest()
console.print(f'Resume: {latest.name if latest else "fresh"}')

In [ ]:
DATA=Path(C.data_dir)

# Try real datasets
USE_BUFFALO=False
try:
    buf=DATA/'buffalo.csv'
    if not buf.exists():
        urllib.request.urlretrieve(
            'http://www.cse.buffalo.edu/~chuangus/Keystroke/DSL-StrongPasswordData.csv',
            buf, )
    USE_BUFFALO=True; console.print('[green]✅ Buffalo dataset[/]')
except Exception as e: console.print(f'[yellow]Buffalo: {e}[/]')

USE_AALTO=False
try:
    aalto=DATA/'aalto.zip'
    if not aalto.exists():
        urllib.request.urlretrieve('https://userinterfaces.aalto.fi/typing37k/data/keystrokes.zip',aalto)
    if not (DATA/'aalto').exists():
        with zipfile.ZipFile(aalto) as z: z.extractall(DATA/'aalto')
    USE_AALTO=True; console.print('[green]✅ Aalto dataset[/]')
except Exception as e: console.print(f'[yellow]Aalto: {e}[/]')

In [ ]:
def gen_seq(T,cognitive=False,motor=False,sev=0.0):
    base_iki=150+sev*220; iki_std=40+sev*140
    dwell_m=80+sev*70; dwell_s=20+sev*50
    err_rate=0.02+sev*0.15
    if motor: base_iki+=sev*100; iki_std+=sev*80

    seq=np.zeros((T,7),np.float32); rol=[]
    for t in range(T):
        iki=max(20,np.random.lognormal(np.log(max(20,base_iki)),iki_std/max(20,base_iki)))
        dw=max(10,np.random.normal(dwell_m,dwell_s))
        fl=max(0,iki-dw); kc=np.random.uniform(0,1)
        is_err=float(np.random.random()<err_rate)
        rol.append(iki)
        if len(rol)>10: rol.pop(0)
        iki_z=(iki-np.mean(rol))/(np.std(rol)+1e-8) if len(rol)>3 else 0.0
        wpm=60000/(np.mean(rol)*5+1e-8)
        seq[t]=[iki/1000,dw/1000,fl/1000,kc,iki_z,wpm/100,is_err]
    return seq

records=[]

# Real data
if USE_BUFFALO:
    try:
        df_buf=pd.read_csv(DATA/'buffalo.csv')
        tcols=[c for c in df_buf.columns if c.startswith(('H.','UD.','DD.'))]
        for subj,sg in df_buf.groupby('subject'):
            for _,row in sg.iterrows():
                ts=row[tcols].values.astype(np.float32) if tcols else np.zeros(C.n_feat,np.float32)
                seq=np.zeros((C.seq_len,C.n_feat),np.float32)
                n=min(C.seq_len,len(ts))
                seq[:n,0]=np.abs(ts[:n]); seq[:n,1]=np.abs(ts[:n])*0.5
                records.append({'seq':seq,'cognitive_decline':0,'motor_degradation':0,
                                'severity':0.0,'stress_level':0.1,'bp_systolic':0.6,'hrv_sdnn':0.45,'cognitive_load':0.2})
    except Exception as e: console.print(f'[yellow]Buffalo parse: {e}[/]')

# Synthetic — normal
for _ in range(5000):
    sev=np.random.uniform(0,0.15)
    records.append({'seq':gen_seq(C.seq_len,False,False,sev),
                    'cognitive_decline':0,'motor_degradation':0,'severity':sev,
                    'stress_level':np.random.uniform(0,0.3),'bp_systolic':np.random.uniform(0.5,0.65),
                    'hrv_sdnn':np.random.uniform(0.4,0.6),'cognitive_load':np.random.uniform(0.1,0.4)})

# Synthetic — cognitive decline
for _ in range(3000):
    sev=np.random.beta(2,2)
    records.append({'seq':gen_seq(C.seq_len,True,False,sev),
                    'cognitive_decline':1,'motor_degradation':0,'severity':float(sev),
                    'stress_level':np.random.uniform(0.2,0.7),'bp_systolic':np.random.uniform(0.55,0.8),
                    'hrv_sdnn':np.random.uniform(0.2,0.45),'cognitive_load':np.random.uniform(0.5,0.9)})

# Synthetic — motor degradation
for _ in range(2000):
    sev=np.random.uniform(0.3,1.0)
    records.append({'seq':gen_seq(C.seq_len,False,True,sev),
                    'cognitive_decline':0,'motor_degradation':1,'severity':float(sev),
                    'stress_level':np.random.uniform(0.1,0.5),'bp_systolic':np.random.uniform(0.55,0.75),
                    'hrv_sdnn':np.random.uniform(0.25,0.5),'cognitive_load':np.random.uniform(0.3,0.7)})

random.shuffle(records)
n_tr=int(0.85*len(records))
tr_recs,vl_recs=records[:n_tr],records[n_tr:]

# Normalise
tr_seqs=np.stack([r['seq'] for r in tr_recs])
scaler=StandardScaler().fit(tr_seqs.reshape(-1,C.n_feat))
np.save(Path(C.out_dir)/'key_mean.npy',scaler.mean_)
np.save(Path(C.out_dir)/'key_std.npy', scaler.scale_)
def norm(seq): return scaler.transform(seq.reshape(-1,C.n_feat)).reshape(seq.shape).astype(np.float32)

console.print(f'[bold]Records: {len(records)} | Train:{n_tr} Val:{len(records)-n_tr}[/]')

In [ ]:
class KeyDataset(Dataset):
    def __init__(self,recs,is_train=True):
        self.recs=recs; self.is_train=is_train
    def __len__(self): return len(self.recs)
    def __getitem__(self,i):
        rec=self.recs[i]; seq=norm(rec['seq'].copy())
        if self.is_train:
            seq+=np.random.randn(*seq.shape).astype(np.float32)*0.05  # Gaussian jitter
            # Temporal CutMix: zero a random window
            if random.random()<0.25:
                cut=random.randint(8,32); st=random.randint(0,C.seq_len-cut)
                seq[st:st+cut]=0.0
            # Mixup with another sample
            if random.random()<0.2:
                j=random.randint(0,len(self.recs)-1)
                seq2=norm(self.recs[j]['seq'].copy())
                lam=np.random.beta(0.4,0.4)
                seq=lam*seq+(1-lam)*seq2
        labels={
            'cognitive_decline': torch.tensor(rec['cognitive_decline'],dtype=torch.float32),
            'motor_degradation':  torch.tensor(rec['motor_degradation'],dtype=torch.float32),
            'severity':           torch.tensor(rec['severity'],     dtype=torch.float32),
            'stress_level':       torch.tensor(rec['stress_level'], dtype=torch.float32),
            'bp_systolic':        torch.tensor(rec['bp_systolic'],  dtype=torch.float32),
            'hrv_sdnn':           torch.tensor(rec['hrv_sdnn'],     dtype=torch.float32),
            'cognitive_load':     torch.tensor(rec['cognitive_load'],dtype=torch.float32),
        }
        return {'seq':torch.from_numpy(seq),'labels':labels}

tr_ds=KeyDataset(tr_recs,True); vl_ds=KeyDataset(vl_recs,False)
cd_labels=np.array([r['cognitive_decline'] for r in tr_recs])
cls_w=1.0/(np.bincount(cd_labels.astype(int))+1e-8)
sampler=WeightedRandomSampler(cls_w[cd_labels.astype(int)],len(cd_labels))
tr_dl=DataLoader(tr_ds,batch_size=C.batch_size,sampler=sampler, num_workers=4,pin_memory=True)
vl_dl=DataLoader(vl_ds,batch_size=C.batch_size,shuffle=False,  num_workers=4,pin_memory=True)
console.print(f'[bold]Train:{len(tr_ds)} Val:{len(vl_ds)}[/]')

In [ ]:
# ── RMSNorm: compatible with PyTorch < 2.4
# nn.RMSNorm was added in PyTorch 2.4; this fallback works on all versions
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich 2019)."""
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

# Use built-in nn.RMSNorm if PyTorch >= 2.4, else our implementation
import torch
try:
    _RMSNorm = nn.RMSNorm   # PyTorch >= 2.4
except AttributeError:
    _RMSNorm = RMSNorm      # fallback


# ── v2: LLaMA-style encoder with RoPE + SwiGLU

class RoPE(nn.Module):
    """
    Rotary Position Embedding (Su et al. 2021 / LLaMA).
    Encodes absolute and relative position of each keystroke in time.
    """
    def __init__(self,dim,base=10000):
        super().__init__()
        inv_freq=1.0/(base**(torch.arange(0,dim,2).float()/dim))
        self.register_buffer('inv_freq',inv_freq)

    def forward(self,x,seq_len):
        # x: (B,n_heads,T,head_dim)
        t=torch.arange(seq_len,device=x.device).float()
        freqs=torch.outer(t,self.inv_freq)          # (T,dim//2)
        emb=torch.cat([freqs,freqs],-1)              # (T,dim)
        cos=emb.cos().unsqueeze(0).unsqueeze(0)     # (1,1,T,dim)
        sin=emb.sin().unsqueeze(0).unsqueeze(0)

        # Rotate: split x into halves and apply rotation
        x1,x2=x[...,:x.shape[-1]//2],x[...,x.shape[-1]//2:]
        rotated=torch.cat([-x2,x1],-1)
        return x*cos+rotated*sin


class SwiGLU(nn.Module):
    """
    SwiGLU activation (Shazeer 2020 / LLaMA FFN).
    FFN(x) = (W1*x * sigmoid(W1*x)) * W2
    Replaces standard GELU — used in LLaMA, PaLM, Gemini.
    """
    def __init__(self,d_model,mult=2.67):
        super().__init__()
        h=int(d_model*mult)
        # Round to multiple of 64 for GPU efficiency
        h=(h+63)//64*64
        self.w1=nn.Linear(d_model,h,bias=False)
        self.w2=nn.Linear(d_model,h,bias=False)
        self.w3=nn.Linear(h,d_model,bias=False)
    def forward(self,x):
        return self.w3(F.silu(self.w1(x))*self.w2(x))


class RoPEAttention(nn.Module):
    """Multi-head attention with RoPE positional encoding."""
    def __init__(self,d_model,n_heads,rope_base=10000,dp=0.1):
        super().__init__()
        self.n_heads=n_heads; self.hd=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model,bias=False)
        self.out=nn.Linear(d_model,d_model,bias=False)
        self.drop=nn.Dropout(dp)
        self.rope=RoPE(self.hd,rope_base)
        self.scale=self.hd**-0.5

    def forward(self,x):
        B,T,D=x.shape
        qkv=self.qkv(x).reshape(B,T,3,self.n_heads,self.hd).permute(2,0,3,1,4)
        q,k,v=qkv[0],qkv[1],qkv[2]  # each (B,H,T,hd)
        # Apply RoPE to q and k
        q=self.rope(q,T); k=self.rope(k,T)
        # Scaled dot-product (SDPA) — uses Flash Attention if available
        out=F.scaled_dot_product_attention(q,k,v,dropout_p=self.drop.p if self.training else 0.0)
        out=out.permute(0,2,1,3).reshape(B,T,D)
        return self.out(out)


class LlamaBlock(nn.Module):
    """Single LLaMA-style transformer block: RMSNorm → RoPE-Attn → SwiGLU."""
    def __init__(self,d_model,n_heads,ffn_mult=2.67,rope_base=10000,dp=0.1):
        super().__init__()
        self.norm1=_RMSNorm(d_model)
        self.attn=RoPEAttention(d_model,n_heads,rope_base,dp)
        self.norm2=_RMSNorm(d_model)
        self.ffn=SwiGLU(d_model,ffn_mult)
        self.drop=nn.Dropout(dp)
    def forward(self,x):
        x=x+self.drop(self.attn(self.norm1(x)))
        x=x+self.drop(self.ffn(self.norm2(x)))
        return x


class KeystrokeTransformerV2(nn.Module):
    """
    LLaMA-style keystroke encoder:
    Input → Linear proj → RoPE + SwiGLU blocks → mean pool → embed
    Multi-task: cognitive decline, motor degradation + regression targets
    Gaussian NLL + Kendall-Gal task weighting
    """
    def __init__(self,cfg:KeyConfig):
        super().__init__()
        self.input_proj=nn.Linear(cfg.n_feat,cfg.d_model)
        # Gradient checkpointing on blocks
        self.blocks=nn.ModuleList([
            LlamaBlock(cfg.d_model,cfg.n_heads,cfg.ffn_mult,cfg.rope_base,cfg.dropout)
            for _ in range(cfg.n_layers)
        ])
        self.final_norm=_RMSNorm(cfg.d_model)
        self.embed_proj=nn.Sequential(
            nn.Linear(cfg.d_model,cfg.embed_dim),nn.GELU(),nn.LayerNorm(cfg.embed_dim)
        )
        self.log_vars=nn.Parameter(torch.zeros(len(ALL_TASKS)))
        self.heads=nn.ModuleDict()
        for t in BINARY_TASKS:
            self.heads[t]=nn.Sequential(nn.Linear(cfg.embed_dim,64),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(64,1))
        for t in REGRESS_TASKS:
            self.heads[t]=nn.Sequential(nn.Linear(cfg.embed_dim,64),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(64,2))

    def forward(self,seq):
        # seq: (B,T,F)
        x=self.input_proj(seq)  # (B,T,d_model)
        # Gradient checkpointing
        if self.training:
            import torch.utils.checkpoint as gc_util
            for block in self.blocks:
                x=gc_util.checkpoint(block,x,use_reentrant=False)
        else:
            for block in self.blocks: x=block(x)
        x=self.final_norm(x)
        embed=self.embed_proj(x.mean(1))  # mean pool
        preds={t:self.heads[t](embed) for t in self.heads}
        return embed,preds,self.log_vars


model=KeystrokeTransformerV2(C).to(DEVICE)
if NUM_GPUS>1: model=nn.DataParallel(model)
try: model=torch.compile(model,mode='reduce-overhead'); console.print('[green]✅ torch.compile[/]')
except: pass
console.print(f'Params:{sum(p.numel() for p in model.parameters())/1e6:.2f}M')

# Sanity
with torch.no_grad():
    dm=torch.randn(2,C.seq_len,C.n_feat).to(DEVICE)
    raw=model.module if hasattr(model,'module') else model
    try: raw=raw._orig_mod
    except: pass
    emb,preds,lvs=raw(dm)
    console.print(f'Embed:{emb.shape} | Tasks:{list(preds.keys())}')

In [ ]:
bce=nn.BCEWithLogitsLoss()
def gnll(pred,tgt):
    mu,lv=pred[:,0],pred[:,1]; var=lv.exp().clamp(1e-6)
    return (0.5*(lv+(mu-tgt)**2/var)).mean()
def kg(loss,lv): return (torch.exp(-lv)*loss+lv).mean()

def compute_loss(preds,labels,log_vars):
    losses=[]
    for i,t in enumerate(BINARY_TASKS):
        losses.append(kg(bce(preds[t].squeeze(-1),labels[t]),log_vars[i]))
    for j,t in enumerate(REGRESS_TASKS):
        losses.append(kg(gnll(preds[t],labels[t]),log_vars[len(BINARY_TASKS)+j]))
    return sum(losses)

class EMA:
    def __init__(self,m,d=0.9995):
        self.d=d
        r=m.module if hasattr(m,'module') else m
        try: r=r._orig_mod
        except: pass
        self.shadow={n:p.data.clone().cpu() for n,p in r.named_parameters() if p.requires_grad}
    def update(self,m):
        r=m.module if hasattr(m,'module') else m
        try: r=r._orig_mod
        except: pass
        for n,p in r.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n]=self.d*self.shadow[n]+(1-self.d)*p.data.cpu()

def get_raw(m):
    r=m.module if hasattr(m,'module') else m
    try: return r._orig_mod
    except: return r

try: opt=bnb.optim.AdamW8bit(model.parameters(),lr=C.lr,weight_decay=C.weight_decay); console.print('[green]✅ AdamW-8bit[/]')
except: opt=torch.optim.AdamW(model.parameters(),lr=C.lr,weight_decay=C.weight_decay)

total_steps=C.epochs*len(tr_dl)//C.grad_accum
sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=C.lr,total_steps=total_steps,pct_start=0.1)
scl=torch.amp.GradScaler("cuda"); ema=EMA(model,C.ema_decay)
start_ep=1; best_val=float('inf'); history=[]

if latest:
    try:
        ck=torch.load(latest,map_location=DEVICE)
        get_raw(model).load_state_dict(ck['model_state'])
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        scl.load_state_dict(ck['scl']); ema.shadow=ck['ema_shadow']
        history=ck['hist']; best_val=ck['best']; start_ep=ck['epoch']+1
        console.print(f'[green]✅ Resumed ep{ck["epoch"]}[/]')
    except Exception as e: console.print(f'[red]Resume fail:{e}[/]')

patience_cnt=0
console.print(f'[bold]Training RoPE+SwiGLU Keystroke: {C.epochs} epochs[/]')

for epoch in range(start_ep,C.epochs+1):
    t0=time.time(); model.train(); tl=0.0
    ap={t:[] for t in BINARY_TASKS}; ay={t:[] for t in BINARY_TASKS}
    opt.zero_grad()
    for step,batch in enumerate(tr_dl):
        seq=batch['seq'].to(DEVICE)
        labels={k:v.to(DEVICE) for k,v in batch['labels'].items()}
        with torch.autocast('cuda',dtype=DTYPE):
            emb,preds,lvs=model(seq)
            loss=compute_loss(preds,labels,lvs)/C.grad_accum
        scl.scale(loss).backward()
        if (step+1)%C.grad_accum==0:
            scl.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),C.grad_clip)
            scl.step(opt); scl.update(); opt.zero_grad(); sch.step(); ema.update(model)
        tl+=loss.item()*C.grad_accum
        for t in BINARY_TASKS:
            ap[t].extend(torch.sigmoid(preds[t].squeeze(-1)).detach().cpu().tolist())
            ay[t].extend(labels[t].cpu().tolist())

    model.eval(); vl=0.0; vp={t:[] for t in BINARY_TASKS}; vy={t:[] for t in BINARY_TASKS}
    with torch.no_grad():
        for batch in vl_dl:
            seq=batch['seq'].to(DEVICE)
            labels={k:v.to(DEVICE) for k,v in batch['labels'].items()}
            with torch.autocast('cuda',dtype=DTYPE):
                emb,preds,lvs=model(seq); vl+=compute_loss(preds,labels,lvs).item()
            for t in BINARY_TASKS:
                vp[t].extend(torch.sigmoid(preds[t].squeeze(-1)).cpu().tolist())
                vy[t].extend(labels[t].cpu().tolist())
    vl/=len(vl_dl)
    aucs={t:roc_auc_score(vy[t],vp[t]) if len(set(vy[t]))>1 else 0.5 for t in BINARY_TASKS}
    history.append({'epoch':epoch,'train_loss':tl/len(tr_dl),'val_loss':vl,**{f'auc_{t}':v for t,v in aucs.items()}})
    console.print(f'Ep{epoch:02d}/{C.epochs}|vl={vl:.4f}|CD={aucs["cognitive_decline"]:.3f}|MD={aucs["motor_degradation"]:.3f}|{time.time()-t0:.0f}s')
    save_ckpt(epoch,get_raw(model).state_dict(),ema.shadow,opt.state_dict(),sch.state_dict(),scl.state_dict(),history,vl)
    if vl<best_val:
        best_val=vl; patience_cnt=0
        torch.save({'model_state':get_raw(model).state_dict(),'ema_shadow':ema.shadow,'aucs':aucs},CKPT_DIR/'key_best.pt')
        console.print('  [green]✅ Best[/]')
    else:
        patience_cnt+=1
        if patience_cnt>=C.patience: console.print('[yellow]Early stop[/]'); break

json.dump(history,open(Path(C.out_dir)/'key_hist.json','w'))
console.print('[bold green]\n✅ Typing model done![/]')

In [ ]:
# Load best + calibration + ONNX export
ck=torch.load(CKPT_DIR/'key_best.pt',map_location=DEVICE)
get_raw(model).load_state_dict(ck['model_state'])
for n,p in get_raw(model).named_parameters():
    if p.requires_grad and n in ck['ema_shadow']: p.data.copy_(ck['ema_shadow'][n].to(DEVICE))
model.eval()

# Calibration
def ece(y,p,n=10):
    bins=np.linspace(0,1,n+1); e=0.0; N=len(y)
    for b in range(n):
        m=(p>=bins[b])&(p<bins[b+1])
        if m.sum()==0: continue
        e+=m.sum()/N*abs(y[m].mean()-p[m].mean())
    return e

vp2={t:[] for t in BINARY_TASKS}; vy2={t:[] for t in BINARY_TASKS}
with torch.no_grad():
    for batch in vl_dl:
        seq=batch['seq'].to(DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            emb,preds,_=model(seq)
        for t in BINARY_TASKS:
            vp2[t].extend(torch.sigmoid(preds[t].squeeze(-1)).cpu().tolist())
            vy2[t].extend(batch['labels'][t].tolist())

fig,axes=plt.subplots(1,len(BINARY_TASKS),figsize=(6*len(BINARY_TASKS),5))
if len(BINARY_TASKS)==1: axes=[axes]
for ax,t in zip(axes,BINARY_TASKS):
    y,p=np.array(vy2[t]),np.array(vp2[t])
    if len(np.unique(y))<2: continue
    fp,mp=calibration_curve(y,p,n_bins=10)
    ax.plot(mp,fp,'s-',label=f'ECE={ece(y,p):.3f}')
    ax.plot([0,1],[0,1],'k--',label='Perfect')
    ax.set_title(f'{t}\nAUC={roc_auc_score(y,p):.3f}')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Keystroke v2 — Reliability Diagrams'); plt.tight_layout()
plt.savefig(Path(C.out_dir)/'key_calibration.png',dpi=150); plt.show()

# ONNX
class KeyExport(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,seq):
        emb,preds,_=self.m(seq)
        bin_s=torch.stack([preds[t].squeeze(-1) for t in BINARY_TASKS],-1)
        reg_s=torch.stack([preds[t][:,0] for t in REGRESS_TASKS],-1)
        return emb,bin_s,reg_s

rm=deepcopy(get_raw(model)).float().cpu().eval()
exp_m=KeyExport(rm).eval()
dummy=torch.randn(1,C.seq_len,C.n_feat)

fp32=Path(C.out_dir)/'key_fp32.onnx'
torch.onnx.export(exp_m,dummy,fp32,
    input_names=['keystroke_seq'],output_names=['key_embedding','binary_risks','regression'],
    dynamic_axes={k:{0:'B'} for k in ['keystroke_seq','key_embedding','binary_risks','regression']},
    opset_version=17,do_constant_folding=True)
onnx.checker.check_model(onnx.load(str(fp32)))

int8=Path(C.out_dir)/'key_int8.onnx'
quantize_dynamic(str(fp32),str(int8),weight_type=QuantType.QInt8,optimize_model=True)

# Training plot
h=pd.DataFrame(history)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(h['epoch'],h['train_loss'],label='Train'); axes[0].plot(h['epoch'],h['val_loss'],label='Val')
axes[0].set_title('Loss (Gaussian NLL + KG)'); axes[0].legend(); axes[0].grid(alpha=0.3)
for t in BINARY_TASKS:
    if f'auc_{t}' in h.columns: axes[1].plot(h['epoch'],h[f'auc_{t}'],label=t)
axes[1].axhline(0.85,color='green',ls='--',label='Target'); axes[1].set_title('AUC'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(Path(C.out_dir)/'key_training.png',dpi=150); plt.show()

tbl=Table(title='Keystroke v2 Results',show_lines=True)
for c in ['Model','FP32 MB','INT8 MB','Speedup']: tbl.add_column(c)
tbl.add_row('RoPE+SwiGLU Transformer',f'{fp32.stat().st_size/1e6:.1f}',f'{int8.stat().st_size/1e6:.1f}',f'{fp32.stat().st_size/int8.stat().st_size:.1f}×')
console.print(tbl)
console.print(f'[bold green]✅ Typing notebook complete! Embed:{C.embed_dim}-d → feeds fusion[/]')